In [ ]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np, pandas as pd, joblib

# Data
base = ['Exp_centered', 'EduOrdinal', 'IsFemale']
X, y = df_model[base].copy(), df_model['Salary_capped'].copy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Pipeline with placeholder
pipe = Pipeline([('poly', PolynomialFeatures(include_bias=False)),
                 ('scaler', StandardScaler()),
                 ('reg', None)])

# Grid search over Ridge & Lasso, degrees 2 & 3
param_grid = [
    {'poly__degree': [2,3], 'reg': [Ridge(max_iter=10000)], 'reg__alpha': np.logspace(-3,3,50)},
    {'poly__degree': [2,3], 'reg': [Lasso(max_iter=10000)], 'reg__alpha': np.logspace(-3,3,50)}
]

grid = GridSearchCV(pipe, param_grid, cv=KFold(5, shuffle=True, random_state=42),
                    scoring='r2', n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

print("\nBest parameters:", grid.best_params_)
print(f"Best CV R²: {grid.best_score_:.4f}")

# Test evaluation
y_pred = grid.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print("\n" + "="*60)
print("FINAL TEST SET PERFORMANCE – BEST REGULARIZED MODEL")
print("="*60)
print(f"R²   : {r2:.4f}\nRMSE : ${rmse:,.2f}\nMAE  : ${mae:,.2f}")
print("\nComparison with Model 3 (from OLS):")
print(f"Model 3 R²  : {res3['R2']:.4f}   RMSE: ${res3['RMSE']:,.2f}")
print(f"Regularized  R²  : {r2:.4f}   RMSE: ${rmse:,.2f}")

# Retrain on all data and save
grid.best_estimator_.fit(X, y)
joblib.dump(grid.best_estimator_, 'best_regularized_model.pkl')
print("\nBest regularized model saved as 'best_regularized_model.pkl'")

# Feature importance
poly = grid.best_estimator_.named_steps['poly']
reg = grid.best_estimator_.named_steps['reg']
features = poly.get_feature_names_out(base)
coefs = reg.coef_
coef_df = pd.DataFrame({'feature': features, 'coef': coefs})
coef_df['abs_coef'] = np.abs(coefs)
print("\nTop 10 most important polynomial features:")
print(coef_df.sort_values('abs_coef', ascending=False).head(10).to_string(index=False))